# Grounding Final Hypotheses - H9 / H10 / H11

**Author**: Konrad Jelen<br>
**Approach**: last hypothesis round against the deployed int8 grounder (bge-reranker int8 + mDeBERTa-v3 SmoothQuant int8 → logistic, OOF macro-F1 0.796 fp32 / 0.795 int8, warm 1.24 s/claim at k=8). Three mechanism-targeting hypotheses - models stay frozen, only softmax channels, aggregation statistics, thresholds and a logistic re-fit are used:

1. **H9 - NLI contradiction channel** (F1): the NLI forward already computes a 3-way softmax; add max-contradiction / max-neutral as logistic features (zero inference cost)
2. **H10 - aggregation beyond max-over-chunks** (F1): distributional features of the per-pair score set - top-2 mean, logsumexp, count above threshold, top1−top2 margin (zero inference cost)
3. **H11 - reranker-first confidence cascade** (latency): run the NLI only when the reranker max falls inside an uncertainty band; out-of-band claims take the reranker-only verdict

All scoring is CPU OpenVINO int8 (the deployed engines). The per-pair score cache over the 111,800 gold (claim, chunk) pairs is built once by `python -m grounding_hypotheses score` (~5 h CPU, `logs/grounding-pair-scoring.log`); this notebook evaluates from the cache. Driver: `src/grounding/grounding_hypotheses.py`.

## Imports

In [ ]:
%load_ext autoreload
%autoreload 2

# Standard library
from pathlib import Path

# Data processing
import numpy as np

# Project
import grounding_hypotheses as gh
from grounding_ensemble import best_macro, counts_at

# Rich console output
from rich import print as rprint

## Configuration

Gates for the round: H9/H10 adopt at OOF macro-F1 >= baseline + 0.010 (visible above the ±0.014 fold noise); H11 quality gate at baseline − one fold-std, plus a measured >= 20% warm-latency saving. The pair cache and OOF protocol (5-fold, seed 42) match the 0.796 baseline.

In [ ]:
PAIRS = gh.PAIRS                 # int8 per-pair score cache (built offline, ~5 h CPU)
BASELINE_MACRO = gh.BASELINE_MACRO  # fp32 2-cross-encoder stack reference
F1_GATE = gh.F1_GATE             # H9/H10 adoption gate
CASCADE_GATE = gh.CASCADE_GATE   # H11 quality gate
BAND = (0.01, 0.66)              # adopted H11 band (from the OOF frontier)

rprint(f"""[white]Configuration[/white]
  Pair cache: [cyan]{PAIRS.relative_to(gh.ROOT)}[/cyan] ([green]{'present' if PAIRS.exists() else 'MISSING - run the score step'}[/green])
  Baseline macro-F1: [yellow]{BASELINE_MACRO}[/yellow]
  H9/H10 gate: [yellow]{F1_GATE}[/yellow]   H11 quality gate: [yellow]{CASCADE_GATE}[/yellow]
  H11 band: [yellow]{BAND}[/yellow]
""")

## Feature build

Loads the pair cache and builds all candidate per-record features: the baseline `{rr_max, nli_ent_max}`, the H9 contradiction / neutral channels, and the H10 distributional aggregations per model.

In [ ]:
feats, y, langs = gh.build_features()
rprint(f"records [yellow]{len(y)}[/yellow]  features [yellow]{len(feats)}[/yellow]  "
       f"hallucination rate [yellow]{(y == 0).mean():.0%}[/yellow]")

## H9 / H10 - feature stacks (OOF macro-F1)

Each row is a logistic over a feature subset, scored out-of-fold (5-fold, seed 42), threshold at the OOF macro-F1 optimum - the same protocol as the 0.796 baseline. The first row re-derives the baseline from the int8 pair cache and must land near 0.795-0.797 (it does), validating the cache.

In [ ]:
rows = gh.eval_stacks(feats, y)
base = next(r for r in rows if r["set"].startswith("baseline"))
lines = ["[white]H9 / H10 ladder[/white]"]
for r in rows:
    d = r["macro"] - base["macro"]
    verdict = "[green]PASS[/green]" if r["macro"] >= gh.F1_GATE else "[indian_red]fail[/indian_red]"
    lines.append(f"  {r['set']:42s} k={r['k']}  macro-F1 [yellow]{r['macro']:.3f}[/yellow]  "
                 f"d [yellow]{d:+.3f}[/yellow]  {verdict}")
rprint("\n".join(lines))

Both F1 hypotheses are **rejected**: H9 adds +0.004/+0.005 and H10 −0.005 to +0.005 - all inside the ±0.014 fold noise. On this gold the unsupported claims are omissions/fabrications rather than contradictions (H9 has little to bite on), and max-over-chunks already extracts what the pair-score distribution knows (H10).

## H11 - reranker-first cascade (quality frontier)

Band sweep over the cached scores: claims whose reranker max falls outside [a, b] take the reranker-only verdict and skip the NLI forward; in-band claims use the stack. Thresholds fit on OOF predictions - no training.

In [ ]:
casc, T_rr, T_st = gh.eval_cascade(feats, y)
ok = [r for r in casc if r["macro"] >= gh.CASCADE_GATE]
lines = [f"[white]H11 frontier[/white]  (T_rr=[yellow]{T_rr:.2f}[/yellow], T_stack=[yellow]{T_st:.2f}[/yellow], gate >= {gh.CASCADE_GATE})"]
for r in sorted(ok, key=lambda r: -r["skip"])[:6]:
    lines.append(f"  band [{r['a']:.2f}, {r['b']:.2f}]  skip [yellow]{r['skip']:.0%}[/yellow]  "
                 f"macro-F1 [yellow]{r['macro']:.3f}[/yellow]  FP {r['fp']}  FN {r['fn']}")
best = max(casc, key=lambda r: r["macro"])
lines.append(f"  macro-optimal: band [{best['a']:.2f}, {best['b']:.2f}]  skip [yellow]{best['skip']:.0%}[/yellow]  "
             f"macro-F1 [yellow]{best['macro']:.3f}[/yellow]")
rprint("\n".join(lines))

## Final benchmark - cascade-adopted grounder vs current

Quality at the adopted band [0.01, 0.66] (OOF, full 2,752 gold) plus the measured warm latency at k=8 on the deployed LATENCY-hint int8 engines (n=150 sampled claims, chunk embeddings cached; `scripts/bench_grounder_cascade.py`, `logs/grounding-cascade-bench.log`). Stage means: pre-filter 38 ms, reranker 577 ms, NLI 569 ms - the cascade removes the NLI stage for decisive claims, so hard claims still pay both models (p90 moves least).

In [ ]:
X = np.column_stack([feats["rr_max"], feats["nli_ent_max"]])
p = gh.oof_p(X, y)
m_base, T, _ = best_macro(p, y)
fp_b, fn_b, _, _ = counts_at(p, y, T)
rr = feats["rr_max"]
_, T_rr, _ = best_macro(rr, y)
out = (rr <= BAND[0]) | (rr >= BAND[1])
flag = np.where(out, rr < T_rr, p < T)
m_casc, fp_c, fn_c = gh._macro_counts(flag, y)

# measured warm per-claim ms (scripts/bench_grounder_cascade.py, n=150, k=8, LATENCY hint)
LAT = {"base": (1184, 1152, 1483), "casc": (857, 759, 1280)}

rprint(f"""[white]Final benchmark - vs current (always-both)[/white]
  [dim]{'-' * 64}[/dim]
  [bold]Quality (OOF, full gold, int8)[/bold]
    macro-F1: [yellow]{m_base:.3f}[/yellow] -> [yellow]{m_casc:.3f}[/yellow]  [dim](d {m_casc - m_base:+.3f}, fold noise +/-0.014)[/dim]
    FP / FN:  [yellow]{fp_b}[/yellow] / [yellow]{fn_b}[/yellow] -> [yellow]{fp_c}[/yellow] / [yellow]{fn_c}[/yellow]
    NLI forwards skipped: [green]{out.mean():.0%}[/green] [dim](OOF; 57% on the serving sample)[/dim]

  [bold]Warm latency, k=8 (measured)[/bold]
    mean:   [yellow]{LAT['base'][0]}[/yellow] -> [green]{LAT['casc'][0]} ms[/green]  [dim](-28%)[/dim]
    median: [yellow]{LAT['base'][1]}[/yellow] -> [green]{LAT['casc'][1]} ms[/green]  [dim](-34%)[/dim]
    p90:    [yellow]{LAT['base'][2]}[/yellow] -> [green]{LAT['casc'][2]} ms[/green]  [dim](-14%)[/dim]

  [bold]Unchanged[/bold]
    footprint [yellow]1.46 GB[/yellow] (same three int8 IRs) | models frozen, thresholds only
""")

## Summary

- **H9 (contradiction channel) - rejected**: +0.004/+0.005 macro-F1, inside fold noise
- **H10 (aggregation beyond max) - rejected**: −0.005 to +0.005, max-over-chunks is sufficient
- **H11 (reranker-first cascade) - adopted**: 61% of claims skip the NLI at macro-F1 0.795 (−0.002, inside noise); measured warm latency mean 1,184 → 857 ms/claim (−28%), median −34%, footprint unchanged, no training

Full ladder and benchmark: `reports/grounding_hypotheses.md`. Serving helper: `cascade_scores` in `src/grounding/grounding_openvino.py`. Research record: `docs/experiments/semantic-grounding-experiments.md` (final hypothesis round) and the updated `semantic-grounding-sota.md` pipeline/latency sections.